In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [3]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"daily": ["weather_code", "uv_index_max", "uv_index_clear_sky_max", "daylight_duration", "sunshine_duration", "sunset", "sunrise", "temperature_2m_max", "temperature_2m_min", "apparent_temperature_max", "apparent_temperature_min", "precipitation_probability_max", "precipitation_hours", "wind_gusts_10m_max", "wind_speed_10m_max"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "precipitation_probability", "rain", "showers", "weather_code", "cloud_cover", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m"],
	"models": "gfs_seamless",
	"timezone": "America/Los_Angeles",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
	"temperature_unit": "fahrenheit",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 52.54148864746094°N 13.359375°E
Elevation: 38.0 m asl
Timezone: b'America/Los_Angeles'b'GMT-7'
Timezone difference to GMT+0: -25200s


In [4]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(3).ValuesAsNumpy()
hourly_rain = hourly.Variables(4).ValuesAsNumpy()
hourly_showers = hourly.Variables(5).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(6).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(10).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["rain"] = hourly_rain
hourly_data["showers"] = hourly_showers
hourly_data["weather_code"] = hourly_weather_code
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Hourly data
                          date  temperature_2m  relative_humidity_2m  \
0   2026-07-02 00:00:00-07:00       69.095299                  62.0   
1   2026-07-02 01:00:00-07:00       71.795303                  55.0   
2   2026-07-02 02:00:00-07:00       74.495300                  49.0   
3   2026-07-02 03:00:00-07:00       77.195297                  41.0   
4   2026-07-02 04:00:00-07:00       78.995300                  37.0   
..                        ...             ...                   ...   
163 2026-07-08 19:00:00-07:00       50.465302                  72.0   
164 2026-07-08 20:00:00-07:00       50.555302                  72.0   
165 2026-07-08 21:00:00-07:00       52.355301                  68.0   
166 2026-07-08 22:00:00-07:00       55.145302                  62.0   
167 2026-07-08 23:00:00-07:00       58.295300                  56.0   

     precipitation  precipitation_probability  rain  showers  weather_code  \
0              0.0                        0.0   0.0    

In [5]:
# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_uv_index_max = daily.Variables(1).ValuesAsNumpy()
daily_uv_index_clear_sky_max = daily.Variables(2).ValuesAsNumpy()
daily_daylight_duration = daily.Variables(3).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(4).ValuesAsNumpy()
daily_sunset = daily.Variables(5).ValuesInt64AsNumpy()
daily_sunrise = daily.Variables(6).ValuesInt64AsNumpy()
daily_temperature_2m_max = daily.Variables(7).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(8).ValuesAsNumpy()
daily_apparent_temperature_max = daily.Variables(9).ValuesAsNumpy()
daily_apparent_temperature_min = daily.Variables(10).ValuesAsNumpy()
daily_precipitation_probability_max = daily.Variables(11).ValuesAsNumpy()
daily_precipitation_hours = daily.Variables(12).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(13).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(14).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["weather_code"] = daily_weather_code
daily_data["uv_index_max"] = daily_uv_index_max
daily_data["uv_index_clear_sky_max"] = daily_uv_index_clear_sky_max
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["sunset"] = daily_sunset
daily_data["sunrise"] = daily_sunrise
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["apparent_temperature_max"] = daily_apparent_temperature_max
daily_data["apparent_temperature_min"] = daily_apparent_temperature_min
daily_data["precipitation_probability_max"] = daily_precipitation_probability_max
daily_data["precipitation_hours"] = daily_precipitation_hours
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Daily data
                        date  weather_code  uv_index_max  \
0 2026-07-02 00:00:00-07:00          53.0          6.85   
1 2026-07-03 00:00:00-07:00           3.0          6.65   
2 2026-07-04 00:00:00-07:00          53.0          6.55   
3 2026-07-05 00:00:00-07:00          51.0          5.95   
4 2026-07-06 00:00:00-07:00          55.0          4.90   
5 2026-07-07 00:00:00-07:00          61.0          6.00   
6 2026-07-08 00:00:00-07:00          51.0          1.35   

   uv_index_clear_sky_max  daylight_duration  sunshine_duration      sunset  \
0                    6.85       60208.320312       45825.269531  1783020752   
1                    6.65       60134.210938       55210.691406  1783107126   
2                    6.70       60053.777344       36239.445312  1783193497   
3                    6.60       59967.203125       41964.378906  1783279865   
4                    6.65       59874.691406       30907.671875  1783366230   
5                    6.55       59776.45